# 관제탑 (control tower)

`configs/*.env` 를 **그대로** 읽고 쓴다. 셸 스크립트(`scripts/*.sh`)도 같은 파일을 읽으므로 둘이 어긋나지 않는다 (`tests/test_config_parity.py`, `tests/test_notebook.py` 가 확인).

- 이 노트북은 **제출하지 않는다**. 만든 sbatch 명령을 복사해 터미널에서 직접 실행한다.
- KISTI: 작업 제출은 `/scratch/$USER` 아래에서만 된다(명령에 `cd` 가 들어 있다).

In [ ]:
import os, sys, json
REPO = os.path.abspath(os.getcwd())
sys.path.insert(0, REPO)
from mslm import tower
CONFIG = 'configs/mc_50m.env'   # 바꿀 config
OVERRIDES = {}                  # 이번 한 번만 바꿀 값, 예: {'SEED': '1', 'MAX_TOKENS': '1e9'}
tower.list_configs()

## 1. 변수 보기
`from` 열: base(기본값) / config(이 파일이 바꿈) / override(OVERRIDES).

In [ ]:
tower.show(CONFIG, OVERRIDES)

## 2. 변수 바꾸기
- 파일에 영구 반영: `tower.set_var(CONFIG, 'MAX_TOKENS', '20e9')` (셸 스크립트도 바로 이 값을 쓴다)
- 이번 제출에만: 위 `OVERRIDES` 에 넣는다 → 아래 명령에 `KEY=value` 로 붙는다.

In [ ]:
# tower.set_var(CONFIG, 'MAX_TOKENS', '20e9')
tower.show(CONFIG, OVERRIDES, section='train')

## 3. 아이디어(모듈)별 구현 코드
MC 층은 다섯 부품으로 조립된다. 새 아이디어는 `mslm/models/registry.py` 에 `@register(kind, name)` 으로 등록하고 config 변수(`MC_DESCRIPTOR`, `MC_ROUTER`, `MC_GATE`, `MC_CONNECT`, `MC_AUX_LOSS`)를 그 이름으로 바꾼다.

In [ ]:
print('사용 중:', tower.parts(CONFIG, OVERRIDES))
print('등록됨:', tower.available_parts())

In [ ]:
for kind, name in tower.parts(CONFIG, OVERRIDES).items():
    print('=' * 20, kind, '=', name)
    print(tower.source_of(kind, name))

## 4. 실행 명령 미리 보기 (직접 복사해서 실행)

In [ ]:
for job in ['prepare_data', 'prepare_eval', 'train', 'eval']:
    print(f'# {job}')
    print(tower.sbatch_command(CONFIG, job, OVERRIDES))
print('# GPU 노드에서 바로(대화형):')
print(tower.shell_command(CONFIG, 'train', OVERRIDES))

## 5. 연기 시험(smoke)
CPU: tiny 크기로 몇 step 학습 → 저장·재로드 일치 → oracle 라우팅 확인. 몇 초면 끝난다.

In [ ]:
tower.smoke(CONFIG, {**OVERRIDES, 'MODEL_SIZE': 'tiny', 'MC_CHUNK': '16', 'VOCAB_SIZE': '256'})

단일 GPU: **GPU 가 할당된 노드에서만** `RUN_GPU = True` 로 바꿔 실행한다 (KISTI MyKSC Jupyter 또는 salloc 노드. 연구실 greenbeard 에서는 노드에서 직접 GPU 사용 금지 → sbatch 사용).

In [ ]:
RUN_GPU = False
if RUN_GPU:
    print(tower.smoke(CONFIG, OVERRIDES, device='cuda', steps=3, seq_len=2048))

## 6. 진행 상황

In [ ]:
print(tower.run_dir(CONFIG, OVERRIDES))
for r in tower.train_log(CONFIG, OVERRIDES, last=10):
    print(r)
for name, s in tower.eval_summaries(CONFIG, OVERRIDES).items():
    print(name, json.dumps(s['results'], ensure_ascii=False)[:2000])